## HOMEWORK 3

## PART 1 - Data Warehouse Modeling: Star Schema

### 1.1 Define Business Process and Fact Grain 

The business process that I'm analyzing is the financial transaction (buy, sell or divident operations)  executed on a trading account. 

The fact grain is what each row represent, in this case one row in Fact_Transaction represents one transaction from the account statement file. 

### 1. 2 Identify Fact and Dimensions 

- Fact transaction : IDTransaction, Unit + all foreign keys (transaction_type_key, time_key, geo_key and symbol_key)

the dimensions: 

- Dim of Transaction Type: transaction_type_key and Transaction Type 
- Dim. of Time: time_key, Date, day, month, quarter, year 
- Dim. Geography: geo_key, Country, Sub- Region and Region 
- DIm. of Symbol: symbol_key, Symbol, Company Name, Sector, Industry


#### Fact Table: Fact_Transaction

| Attribute | Type | Description|
| ---| --- | ---|
| IDTransaction| Natural key| Unique identifier of transaction|
| Unit| Measure| Number of shares involved in a transaction|
| transaction_type_key| Foreign key| Dim. TransactionType|
| time_key| Foreign key| Dim. Time|
| geo_key | Foreign key | Dim. Geographic|
| symbol_key| Foreign key| Dim. Symbol|


#### Dimenson of TransactionType

| Attribute | Type | Description|
| ---| --- | ---|
| transaction_type_key| Surrogate key| Primary key|
| TransactionType| Descriptive| Buy/ sell/ divident| 


#### Dimension of Time 

| Attribute | Type | Description|
| ---| --- | ---|
| time_key| Surrogate key| Primary key|
| Date| Descriptive| Full date of transaction| 
| Day| Descriptive| Day of the month|
| Month | Descriptive | Month number |
| Quarter | Descriptive | Quarter (Q1–Q4) |
| Year | Descriptive | Year |


#### Geografic Dimension

| Attribute | Type | Description |
|---|---|---|
| geo_key | Surrogate key| Primay key|
| Country | Descriptive | Country name |
| Sub_Region | Descriptive | Geographic sub-region |
| Region | Descriptive | Geographic region |


#### Symbol Dimension

| Attribute | Type | Description |
|---|---|---|
| symbol_key | Surrogate key| Primary key|
| Symbol | Descriptive | Stock ticker symbol |
| Company_Name | Descriptive | Full company name |
| Sector | Descriptive | Market sector |
| Industry | Descriptive | Specific industry |


### 1.3 Define Dimension Hierarchies

- Dimension Time: Day → Month → Quarter → Year
- DImension Geographic: Country → Sub_Region → Region
- DImension Attribute: Symbol → Industry → Sector
- Dimenstion Transaction Type: no hierarchy, it just have descriptive attributes

### 1.4 Star Schema

![Star_Schema](Star_Schema.png)

## PART 2 - Data Tranformation and Analysis

In [2]:
# First i import the libraries that i will need for the analysis and visualization of data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### 2.1 Load and Clean the Data

In [3]:
# then i upload the dataset with different name, 

df_account_raw = pd.read_csv("data/Datasets/account-statement-1-1-2024-12-31-2024.csv", sep=";")
df_country_raw = pd.read_csv("data/Datasets/country.csv")
df_symbols_raw = pd.read_csv("data/Datasets/symbols.csv", sep=";")

df_account= df_account_raw.copy()
df_country= df_country_raw.copy()
df_symbols= df_symbols_raw.copy()

print("Account shape:", df_account.shape)
print("Country shape:", df_country.shape)
print("Symbols shape:", df_symbols.shape)
print() 

df_account= df_account[["IDTransaction", "Date", "TransactionType", "Symbol", "Unit"]]
df_country= df_country[["name", "sub-region", "region"]]
df_symbols= df_symbols[["symbol", "company_name", "sector", "industry", "country"]]

display(df_account.head(3))
display(df_country.head(3))
display(df_symbols.head(3))

Account shape: (2745, 6)
Country shape: (249, 11)
Symbols shape: (3194, 5)



,IDTransaction,Date,TransactionType,Symbol,Unit
0,2.769834e+09,11/01/2024 10:44:03,BUY,BAP,1605.0
1,2.767325e+09,24/01/2024 08:07:24,SELL,BAP,1605.0
2,2.815474e+09,10/01/2024 11:00:08,SELL,BAP,914.0


,name,sub-region,region
0,Afghanistan,Southern Asia,Asia
1,Åland Islands,Northern Europe,Europe
2,Albania,Southern Europe,Europe


,symbol,company_name,sector,industry,country
0,TEAM,Atlassian Corporation,Technology,Software - Application,Australia
1,WDS,Woodside Energy Group Limited,Energy,Oil & Gas E&P,Australia
2,OSW,OneSpaWorld Holdings Limited,Consumer Cyclical,Leisure,Bahamas


In [4]:
# To visualize the data i will not use head since it show just the firs 5, instead i will use a summary
# Here we inspect column names, data types and missing values.

print("Account summary")
summary_acc = pd.DataFrame({
    "column": df_account.columns,
    "dtype": df_account.dtypes.astype(str).values,
    "missing_values": df_account.isna().sum().values,
    "missing_pct": (df_account.isna().mean().values * 100).round(2)
})

display(summary_acc)


print("Country summary")
summary_co = pd.DataFrame({
    "column": df_country.columns,
    "dtype": df_country.dtypes.astype(str).values,
    "missing_values": df_country.isna().sum().values,
    "missing_pct": (df_country.isna().mean().values * 100).round(2)
})

display(summary_co)


print("Symbol summary")

summary_sy = pd.DataFrame({
    "column": df_symbols.columns,
    "dtype": df_symbols.dtypes.astype(str).values,
    "missing_values": df_symbols.isna().sum().values,
    "missing_pct": (df_symbols.isna().mean().values * 100).round(2)
})

display(summary_sy)

Account summary


,column,dtype,missing_values,missing_pct
0,IDTransaction,float64,464,16.9
1,Date,str,464,16.9
2,TransactionType,str,464,16.9
3,Symbol,str,464,16.9
4,Unit,float64,464,16.9


Country summary


,column,dtype,missing_values,missing_pct
0,name,str,0,0.0
1,sub-region,str,2,0.8
2,region,str,2,0.8


Symbol summary


,column,dtype,missing_values,missing_pct
0,symbol,str,0,0.0
1,company_name,str,0,0.0
2,sector,str,0,0.0
3,industry,str,0,0.0
4,country,str,0,0.0


In [5]:
# From this we can see that df_account has 464 rows in all columns and the column "Unnamed:5 " is an empty column.
# According to what we need for the anlysis, country has just 1 missing value in "Alpha-2" and 2 missin value in region and
# sub region. 
# While, Symbols is the perfect dataset with zero missing values. 

# I procede to remove all missing values in df_account
df_account = df_account.dropna(subset=["IDTransaction"])

# Actually the column Unnamd was written in the previous analysis, and it give an error if i delet it, so i just put the instruction 
# to ingnore te error 
df_account = df_account.drop(columns=["Unnamed: 5"], errors='ignore')

print("Clean Account Dataset:", df_account.shape)

# I select the columns that I will need for this analysis
df_country= df_country[["name","sub-region" , "region"]]
print("\nClean Country Dataset", df_country.shape)
print(df_country.isnull().sum())

# I visualize the Symbols dataset 
print("\ndf_symbols: already clean")

Clean Account Dataset: (2281, 5)

Clean Country Dataset (249, 3)
name          0
sub-region    2
region        2
dtype: int64

df_symbols: already clean


In [6]:
# Inspect rows with NaN in df_country to verify what are the missing countries
print(df_country[df_country.isnull().any(axis=1)])

                          name sub-region region
8                   Antarctica        NaN    NaN
217  Taiwan, Province of China        NaN    NaN


In [7]:
# Check if these countries appear in df_symbols because it's important for the analisys
miss_country = ["Antarctica", "Namibia", "Taiwan, Province of China"]
print(df_symbols[df_symbols["country"].isin(miss_country)])

# And we can see that the rows with NaN in df_country do not appear in df_symbols

Empty DataFrame
Columns: [symbol, company_name, sector, industry, country]
Index: []


In [8]:
# Verify that every transaction symbol exists in the symbols datase without duplicates
symbols_in_account = set(df_account["Symbol"].dropna().unique())
symbols_in_symbols = set(df_symbols["symbol"].unique())

missing_symbols = symbols_in_account - symbols_in_symbols
print(f"Symbols in account but NOT in symbols dataset: {missing_symbols}")

# Verify that every company country can be mapped to the country dataset
countries_in_symbols = set(df_symbols["country"].unique())
countries_in_country = set(df_country["name"].unique())

missing_countries = countries_in_symbols - countries_in_country
print(f"Countries in symbols NOT in country dataset: {missing_countries}")

Symbols in account but NOT in symbols dataset: {'RIGZU', 'VWS', 'OBDC', 'MFG', 'MONC', 'CSIQ', 'AZM', 'IBE', 'FNC', 'SAP', 'RCMT', 'TKC', 'HTGC', 'ARCH', 'UCG', 'CCAP', 'WF', 'AGO.l'}
Countries in symbols NOT in country dataset: {'Turkey', 'Taiwan'}


In [9]:
# The symbols that are not present in the Symbols dataset are 18, but I chek and I saw that they exists
# in the Account dataset but not in the Symbols dataset. 
# For the countries there is a unmatched in the name of the country in the two dataset.

In [10]:
# -------------- Symbols not found -------------------------------
missing_symbols = symbols_in_account - symbols_in_symbols
print(f"Transactions with unmatched symbols: {df_account[df_account['Symbol'].isin(missing_symbols)].shape[0]}")

# ------------- Fix country name mismatches -----------------------------
df_symbols["country"] = df_symbols["country"].replace({
    "Turkey": "Türkiye",
    "Taiwan": "Taiwan, Province of China"
})

# Re-check
missing_countries = set(df_symbols["country"].unique()) - set(df_country["name"].unique())
print(f"Countries still missing after fix: {missing_countries}")

Transactions with unmatched symbols: 212
Countries still missing after fix: set()


In [11]:
# Remove transactions with symbols not in symbols dataset
df_account = df_account[df_account["Symbol"].isin(symbols_in_symbols)]
print(f"df_account after removing unmatched symbols: {df_account.shape}")

df_account after removing unmatched symbols: (2069, 5)


In [12]:
# Then I have to built the different dimensions, so I will start with:

# ------------- Dim Transaction Type---------------------------------

# From the dataset i took all the column TransactionType without duplicates (i just save the "transaction type" of the operation). 
# I have to reset the count because, with this operation, many row will be passed, so i reset the count from 0. 
dim_transaction_type= df_account[["TransactionType"]].drop_duplicates().reset_index(drop= True)

# Then I have to create like a key, so on the dimension of the type of transaction I have
dim_transaction_type.insert(0, "transaction_type_key", range(1, len (dim_transaction_type) +1))

# ------------- Dim Time---------------------------------

# I did the same thing for the dimension of time. From the dataset I split the date into the unit I want.
df_account["Date"]= pd.to_datetime(df_account["Date"], format= "%d/%m/%Y %H:%M:%S")
dim_time= df_account[["Date"]].drop_duplicates().reset_index(drop= True)

dim_time.insert(0, "time_key", range(1, len(dim_time) +1))

dim_time ["Day"]= dim_time["Date"].dt.day
dim_time ["Month"]= dim_time["Date"].dt.month
dim_time ["Quarter"]= dim_time["Date"].dt.quarter
dim_time ["Year"]= dim_time["Date"].dt.year

# I create a list with all temporal keys of 2024
time_keys_2024 = dim_time[dim_time["Year"] == 2024]["time_key"].values

# # ------------- Dim Geography---------------------------------

# In this case, I also remove the duplicates and taking into account the list of countries, I match the "country" on the left to the 
# "name" of the right. By doing this the column of name is dropped there is the duplicate in the country column.
dim_geography= df_symbols[["country"]].drop_duplicates().reset_index(drop= True)
dim_geography= dim_geography.merge(df_country, left_on= "country", right_on= "name", how= "left").drop(columns=["name"])

dim_geography.insert(0, "geo_key", range(1, len(dim_geography) +1))

# ------------- Dim Symbols---------------------------------

# SO, in this case, after the drop of the duplicates, I procede with a "left join" on "country" to match each country from 'symbols' dataset 
# to the country of 'country' dataset on the key of "geo_key". I drop the 'country' column to keep only the foreign key and insert a primary 
# on symbol.
dim_symbols= df_symbols[["symbol", "company_name", "sector", "industry", "country"]].drop_duplicates().reset_index(drop= True)
dim_symbols= dim_symbols.merge(dim_geography[["geo_key", "country"]], on="country", how= "left").drop(columns=["country"])

dim_symbols.insert(0, "symbol_key", range(1, len(dim_symbols) +1))

#--------------------------------------------------------------

print("dim_transaction_type:", dim_transaction_type.shape)
display(dim_transaction_type)

print("dim_time:", dim_time.shape)
display(dim_time.head())

print("dim_geography:", dim_geography.shape)
display(dim_geography.head())

print("dim_symbols:", dim_symbols.shape)
display(dim_symbols.head())

dim_transaction_type: (3, 2)


,transaction_type_key,TransactionType
0,1,BUY
1,2,SELL
2,3,DIVIDENT


dim_time: (2005, 6)


,time_key,Date,Day,Month,Quarter,Year
0,1,2024-01-11 10:44:03,11,1,1,2024
1,2,2024-01-24 08:07:24,24,1,1,2024
2,3,2024-01-10 11:00:08,10,1,1,2024
3,4,2024-01-16 08:14:21,16,1,1,2024
4,5,2024-01-16 14:34:12,16,1,1,2024


dim_geography: (42, 4)


,geo_key,country,sub-region,region
0,1,Australia,Australia and New Zealand,Oceania
1,2,Bahamas,Latin America and the Caribbean,Americas
2,3,Bermuda,Northern America,Americas
3,4,Brazil,Latin America and the Caribbean,Americas
4,5,Virgin Islands (British),Latin America and the Caribbean,Americas


dim_symbols: (3194, 6)


,symbol_key,symbol,company_name,sector,industry,geo_key
0,1,TEAM,Atlassian Corporation,Technology,Software - Application,1
1,2,WDS,Woodside Energy Group Limited,Energy,Oil & Gas E&P,1
2,3,OSW,OneSpaWorld Holdings Limited,Consumer Cyclical,Leisure,2
3,4,ACGL,Arch Capital Group Ltd.,Financial Services,Insurance - Diversified,3
4,5,AGO,Assured Guaranty Ltd.,Financial Services,Insurance - Specialty,3


In [13]:
# Now I have to link the fact table "df_account" to the dimension I create.
# To do this this I do a left join on the key I create.

fact_transaction= df_account.copy()

# For the first, I do the merge on TransactionType that is the core on which the two dimensions should be merged

fact_transaction= fact_transaction.merge(
    dim_transaction_type, on= "TransactionType", how= "left"
)

# For the time, I specify that the keys are two: time_key that is the primary and Date that is the foreign key
fact_transaction= fact_transaction.merge(
    dim_time[["time_key", "Date"]], on= "Date", how= "left"
)

# I did first the symbol dimension, because for the geography dimension I have to take from the symbol dimension
fact_transaction= fact_transaction.merge(
    dim_symbols[["symbol_key", "symbol"]], left_on= "Symbol", right_on= "symbol", how= "left"
)

# for the geography dimension I take the key geo_key through the symbol_key (because there is the country the link the two dataset)
fact_transaction= fact_transaction.merge(
    dim_symbols[["symbol_key", "geo_key"]], on= "symbol_key", how= "left"
)

# For the fact table, moreover I have to select only te columns I need.

fact_transaction= fact_transaction[["IDTransaction", "Unit", "transaction_type_key", "time_key", "symbol_key", "geo_key"]]

fact_transaction["IDTransaction"]= fact_transaction["IDTransaction"].astype(int)
#without this actually I saw that pd automatically show the value as a float (e+09), so I just wrote to show it as an int like it is 

print("fact_transaction:", fact_transaction.shape)
display(fact_transaction)

fact_transaction: (2069, 6)


,IDTransaction,Unit,transaction_type_key,time_key,symbol_key,geo_key
0,2769834124,1605.0,1,1,284,32
1,2767324642,1605.0,2,2,284,32
2,2815473914,914.0,2,3,284,32
3,2622244212,646.0,1,4,4,3
4,2629871124,646.0,2,5,258,26
...,...,...,...,...,...,...
2064,2659221801,45.0,2,2001,197,20
2065,2648833741,45.0,2,2002,1858,42
2066,2738173270,45.0,2,2003,171,9
2067,2680935720,45.0,2,2004,1964,42


### 2.2 Analytical Questions

In [14]:
# I will provide to answer to 5 questions.

# --------------- What are the top 5 sectors by number of SELL transactions in US during 2024? ----------------------------

# For this question I have to link the fact_transaction to the transacion_type_key based on the SELL. 
# So I need the "dim_symbols" to filter on the sector, the "dim_geography" to filter for the United States adnd the "dim_time" to 
# filter for the 2024. 

# transaction_type_key for the SELL 
sell_key= dim_transaction_type[dim_transaction_type["TransactionType"] == "SELL"]["transaction_type_key"].values[0]

# us_key for the Unites States
us_key= dim_geography[dim_geography["country"] == "United States of America"]["geo_key"].values[0]

# ---------------------------

# After set the filter for the sell and the country, I procede with the merge 
q1= fact_transaction [
    (fact_transaction["transaction_type_key"] == sell_key) &
    (fact_transaction["geo_key"] == us_key) &
    (fact_transaction["time_key"].isin(time_keys_2024))
].merge(dim_symbols[["symbol_key", "sector"]], on= "symbol_key", how= "left")

# I grouped by the sector, based on the number of SELL transaction 
q1_top5= q1.groupby("sector").size().reset_index(name="num_transaction")
q1_top5= q1_top5.sort_values("num_transaction", ascending= False).head(5)

print("Top 5 sectors by number of SELL transactions in US during 2024: ")
display(q1_top5)

Top 5 sectors by number of SELL transactions in US during 2024: 


,sector,num_transaction
6,Technology,158
0,Communication Services,58
3,Financial Services,55
4,Healthcare,50
1,Consumer Cyclical,48


In [15]:
# --------------- What are the top 5 industries by number of BUY transactions in Q4 of 2024? -----------------------------

# For the question 2 what we need is link  the fact_transactio to the transaction_type_key, but this time, on the type of BUY. 
# So analyze the top 5 industries, that we take from the "dim_symbols" ordered by number of BUY, in the last quarter of 2024. 

# transaction_type_key for BUY category
buy_key= dim_transaction_type[dim_transaction_type["TransactionType"] == "BUY"]["transaction_type_key"].values[0]

# I set, from the dimension of time, the quarter 
time_keys_q4= dim_time[(dim_time["Quarter"] == 4) & (dim_time["Year"] == 2024)]["time_key"].values

# Then, also in this case I did a merge but this time based on the industries
q2 = fact_transaction [
    (fact_transaction["transaction_type_key"] == buy_key) &
    (fact_transaction["time_key"].isin(time_keys_q4)) 
].merge(dim_symbols[["symbol_key", "industry"]], on="symbol_key", how="left")

# I grouped by the industries, based on the number of BUY transaction 
q2_top5= q2.groupby("industry").size().reset_index(name= "num_transaction")
q2_top5= q2_top5.sort_values("num_transaction", ascending= False).head(5)

print("The top 5 industries by number of BUY transactions in Q4 of 2024:")
display(q2_top5)


The top 5 industries by number of BUY transactions in Q4 of 2024:


,industry,num_transaction
17,Semiconductors,18
10,Internet Content & Information,15
19,Software - Infrastructure,10
11,Internet Retail,8
4,Diagnostics & Research,7


In [16]:
# --------------- Rank all quarters of 2024 by total number of transactions (BUY + SELL) -----------------------------

# For this question I have rank all the quadrant based on the total number of transactions. So what I need are just 
# the quarters ("dim_time") and the TransactionType

# I create a key to consider just the quarter of the year based on "time_key"
time_keys_quarter= dim_time[(dim_time["Year"] == 2024)]["time_key"].values

# I create a key to link buys and sells basedon the "transaction_type_key"
buy_sell_key= dim_transaction_type[
    dim_transaction_type["TransactionType"].isin(["BUY", "SELL"])
]["transaction_type_key"].values

# Then i do the merge based on the quarter of the year 
q3= fact_transaction[
    (fact_transaction["transaction_type_key"].isin(buy_sell_key)) &
    (fact_transaction["time_key"].isin(time_keys_quarter))
].merge(dim_time[["time_key", "Quarter"]], on="time_key", how="left")

# I group the quarter based on the total number of transactions
q3_rank= q3.groupby("Quarter").size().reset_index(name="total_transactions")
q3_rank= q3_rank.sort_values("total_transactions", ascending= False)
print("Quarters ranked by total transactions: ")
display(q3_rank)

Quarters ranked by total transactions: 


,Quarter,total_transactions
0,1,968
1,2,522
2,3,242
3,4,241


In [17]:
# --------------- What are the top 10 countries by number of SELL transactions in 2024? -----------------------------

# What I have to do, is classify the top 10 countries based on the number of SELL transactions in 2024.
# We can reuse the sell_key to determine all the sells

q4= fact_transaction [
    (fact_transaction["transaction_type_key"] == sell_key) &
    (fact_transaction["time_key"].isin(time_keys_2024))
].merge(dim_geography[["geo_key", "country"]], on= "geo_key", how= "left")

# I grouped by the country, based on the number of SELL transaction 
q4_top10= q4.groupby("country").size().reset_index(name="num_transaction")
q4_top10= q4_top10.sort_values("num_transaction", ascending= False).head(10)

print("Top 10 countries by number of SELL transactions in 2024: ")
display(q4_top10)

Top 10 countries by number of SELL transactions in 2024: 


,country,num_transaction
19,United States of America,389
18,United Kingdom of Great Britain and Northern I...,130
4,China,112
1,Brazil,69
17,"Taiwan, Province of China",50
13,"Netherlands, Kingdom of the",46
16,Switzerland,37
8,Ireland,31
10,Luxembourg,27
2,Canada,22


In [18]:
# --------------- What are the top 5 regions by total units bought in 2024? ------------------------------

# In this case we have to analyze the total units bought ("BUY", from the fact_transaction) ordered by the regions.

# Actually we can reuse the "bu_key" from the question 2 and do the merge based on the geo_key
q5= fact_transaction [
    (fact_transaction["transaction_type_key"] == buy_key)& 
    (fact_transaction["time_key"].isin(time_keys_2024))
].merge(dim_geography[["geo_key", "region"]], on= "geo_key", how= "left")

# I grouped by the region, based on the total sum of units bought
q5_top5= q5.groupby("region")["Unit"].sum().reset_index(name= "total_units")
q5_top5= q5_top5.sort_values("total_units", ascending= False).head(5)

print("top 5 regions by total units bought in 2024:")
display(q5_top5)

top 5 regions by total units bought in 2024:


,region,total_units
0,Americas,37026.0
2,Europe,22528.0
1,Asia,9198.0


# Report - Financial Transactions Star Schema and Dashboard

**Data Warehouse Modeling**

The conceptual representation of this project is based on the **Dimensional Fact Model**, which consists of a fact schema representing the core business process and a set of dimensions used to analyze it.

The **business process** analyzed is the management of *financial transactions* (BUY, SELL, and DIVIDEND operations) executed on a trading account involving stocks listed across multiple countries.

The **grain of the fact table** is defined as follows: each row in *Fact_Transactions*represents a single, independent financial transaction recorded in the account statement. This corresponds to an *Event Fact*, since each row captures a punctual event with a specific numerical measure (Unit), with no cumulative or temporal relationship between rows.

The star schema includes one fact table and four dimension tables:

- **Dim_Time** — hierarchy: Day → Month → Quarter → Year
- **Dim_Geography** — hierarchy: Country → Region → Sub-Region
- **Dim_Symbol** — hierarchy: Symbol → Industry → Sector
- **Dim_TransactionType** — no hierarchy applicable (flat dimension with a single 
descriptive attribute)

The star schema was designed using the online platform *"dbdiagram.io"*; I define Fact_Transiction table and the four dimensions linked by the foreign keys. 

-----

**Data Transformation and Analysis**

The three source datasets were *loaded* and filtered to retain only the attributes included in the dimensional model, following the professor's instructions. 
The following data quality checks were performed:

*Missing values:* The account statement dataset contained 464 completely empty rows, which were removed. An additional spurious column (`Unnamed: 5`), generated by a trailing separator in the CSV file, was also dropped. The country dataset presented a small number of missing values in the `region` and `sub-region` columns (Antarctica, Namibia, Taiwan). However, since none of these countries appeared in the symbols dataset, they did not affect the analysis and were retained as-is.

I procede to ***verify that every transactions symbol exist in the symbols dataset*** in which I saw that 18 stock symbols present in the account statement were not found in the symbols dataset. This resulted in 212 unmatched transactions, which were removed from the analysis since they could not be linked to the Dim_Symbol dimension.

And then I ***verify that every company country can be mapped to the country dataset***. Two country name mismatches were identified between the symbols dataset and the country dataset: "Turkey" was renamed to "Türkiye" and "Taiwan" was renamed to "Taiwan, Province of China", in accordance with the ISO 3166-1 standard used in the country dataset. After this correction, all countries were successfully mapped.

----

**Dimensions and Fact Table Construction**

The four dimension tables were built from the cleaned datasets by selecting the relevant attributes, removing duplicates, and assigning surrogate keys. The *Fact_Transactions* table was then constructed by joining the account statement with each dimension by a left joins, using the surrogate keys as foreign keys. This because the Fact_Transaction is the core of our analysis, which is mean that the table from which we start the join is the one at the left, by doing this we take all the columns of the Fact_Transaction and we match it with the dimensions, if there's the match. if there's not the corresponding match then it remain as a NaN or Null value.

Then I procede to answer **5 business questions:**

- Top 5 sectors by number of SELL transactions in the US during 2024. 
The leading sector is *Technology* with 158 transactions, followed by *Communication Services* (58), *Financial Services* (55), *Healthcare* (50), and *Consumer Cyclical* (48). The dominance of the Technology sector reflects the high trading volume of US-listed tech stocks.

- Top 5 industries by number of BUY transactions in Q4 2024. 
The most actively bought industry was *Semiconductors* (18 transactions), followed by *Internet Content & Information* (15), *Software - Infrastructure* (10), *Internet Retail* (8), and *Diagnostics & Research (7).*

- Quarters ranked by total transactions (BUY + SELL). 
The quarters are ranked in chronological order *(Q1, Q2, Q3, Q4)*, suggesting a relatively uniform distribution of trading activity throughout 2024.

- Top 10 countries by number of SELL transactions in 2024. 
The *United States of America* ranked first by a large margin with 389 SELL transactions. Mid-ranking countries include *Taiwan - Province of China* (50) and *Netherlands* (46). *Canada* ranked tenth with 22 transactions.

- Top 5 regions by total units bought in 2024. 
The three dominant regions are *Americas*, *Europe*, and *Asia*, reflecting the geographic distribution of the stocks present in the portfolio.

---

**Streamlit Dashboard**

An interactive dashboard was developed using *Streamlit* in a dedicated ***"app.py"*** file. The datasets are loaded and cleaned at startup, replicating the ETL logic from the notebook.

***Time Analysis*** includes a date range filter with default values from 01/01/2024 to 31/12/2024. All four charts react dynamically to the selected date range: a *line chart* showing the total number of transactions over time, a *bar chart* of the top 3 traded symbols, a bar chart of the top 5 sectors, and a bar chart of the top 5 industries by transaction count.